In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dropout, GlobalMaxPooling1D, Dense
from tensorflow.keras.callbacks import EarlyStopping

## 1. Load and Preprocess Data

In [ ]:
# Load dataset
DATA_PATH = 'Dataset_5971.csv'
df = pd.read_csv(DATA_PATH)

# Clean slightly
df = df.rename(columns={"LABEL": "Target", "TEXT": "Text"})
df['Text'] = df['Text'].fillna("").apply(lambda x: x.lower())

# Check unique labels
unique_labels = df['Target'].unique()
num_classes = len(unique_labels)
print(f"Detected labels: {unique_labels} ({num_classes} classes)")

# Tokenization
max_vocab = 7000
max_len = 500

tokenizer = Tokenizer(num_words=max_vocab)
tokenizer.fit_on_texts(df['Text'])

sequences = tokenizer.texts_to_sequences(df['Text'])
X = pad_sequences(sequences, maxlen=max_len, padding='post')

# Encode Labels (0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(df['Target'])
label_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print(f"Label Mapping: {label_mapping}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to handle imbalance
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Final Training shape: {X_train_res.shape}")

Detected labels: ['ham' 'Smishing' 'spam' 'Spam' 'smishing'] (5 classes)
Label Mapping: {np.int64(0): 'Smishing', np.int64(1): 'Spam', np.int64(2): 'ham', np.int64(3): 'smishing', np.int64(4): 'spam'}
Final Training shape: (19350, 500)


## 2. Build GRU Model (Multi-Class Fix)

In [ ]:
import tensorflow as tf

if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
else:
    print("No GPU found. Please install GPU version of TensorFlow for optimal performance.")

model = Sequential([
    Embedding(input_dim=max_vocab, output_dim=128, input_length=max_len),
    Bidirectional(GRU(128, return_sequences=True)),
    Dropout(0.3),
    GRU(64, return_sequences=True),
    Dropout(0.3),
    GlobalMaxPooling1D(),
    # --- CHANGE: Output 3 neurons with softmax for 3 classes ---
    Dense(num_classes, activation='softmax')
])

# --- CHANGE: Use sparse_categorical_crossentropy for integer labels ---
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Default GPU Device: /device:GPU:0


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 3. Train Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train_res, y_train_res,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    batch_size=64
)

Epoch 1/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 27s 65ms/step - accuracy: 0.4139 - loss: 1.3201 - val_accuracy: 0.7916 - val_loss: 0.6909
Epoch 2/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 19s 64ms/step - accuracy: 0.7101 - loss: 0.7473 - val_accuracy: 0.7841 - val_loss: 0.6708
Epoch 3/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 20s 65ms/step - accuracy: 0.7957 - loss: 0.5554 - val_accuracy: 0.8552 - val_loss: 0.4381
Epoch 4/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 20s 66ms/step - accuracy: 0.8512 - loss: 0.4106 - val_accuracy: 0.8251 - val_loss: 0.5382
Epoch 5/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 19s 64ms/step - accuracy: 0.8960 - loss: 0.3004 - val_accuracy: 0.8536 - val_loss: 0.4611


## 4. Evaluation and Prediction (Multi-Class Logic)

In [ ]:
def predict_sms(text):
    seq = tokenizer.texts_to_sequences([text.lower()])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    # Get probabilities for all 3 classes
    preds = model.predict(pad)[0]
    predicted_index = np.argmax(preds)
    label = le.inverse_transform([predicted_index])[0]
    confidence = preds[predicted_index]

    print(f"Message: {text}")
    print(f"Prediction: {label} ({confidence:.2%})")

predict_sms("WIN a FREE iPhone! Call 9876543210 immediately.")
predict_sms("Hey, are we still meeting today at 5?")
predict_sms("Urgent: Your account is blocked. Click here: http://bit.ly/fake-link")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
Message: WIN a FREE iPhone! Call 9876543210 immediately.
Prediction: Smishing (45.02%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Message: Hey, are we still meeting today at 5?
Prediction: ham (97.96%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Message: Urgent: Your account is blocked. Click here: http://bit.ly/fake-link
Prediction: Smishing (88.01%)
